# Semantic Categories #

This Notebook will be used to check our available labels, and decide on a set of semantic categories that will be used for training. 

In [16]:
import numpy as np
import nltk
from nltk.corpus import wordnet as wn
from collections import defaultdict

# Download WordNet data (only needed once)
try:
    wn.synsets('test')
except LookupError:
    print("Downloading WordNet data...")
    nltk.download('wordnet')
    nltk.download('omw-1.4')
    print("✓ Download complete")

### WordNet Basics - Exploring Synsets and Hypernyms

Let's explore how WordNet represents words and their hierarchies

In [15]:
# Example 1: Single word - "dog"
word = "dog"
synsets = wn.synsets(word, pos=wn.NOUN)  # Get all noun synsets for "dog"

print(f"Word: '{word}'")
print(f"Number of synsets (meanings): {len(synsets)}\n")

# Let's use the first (most common) synset
synset = synsets[0]
print(f"Synset: {synset}")
print(f"Definition: {synset.definition()}")
print(f"Examples: {synset.examples()}")
print(f"\nHypernym path (from specific to general):")

# Get hypernym path - shows the hierarchy
paths = synset.hypernym_paths()
for i, path in enumerate(paths[0]):  # Show first path
    print(f"  {i}. {path.name().split('.')[0]:20s} - {path.definition()[:60]}")

NameError: name 'wn' is not defined

In [14]:
# Example 2: Multi-word term from our data - "Tibetan_mastiff"
word = "Tibetan_mastiff"

# Try as-is
synsets = wn.synsets(word, pos=wn.NOUN)
print(f"Searching for: '{word}'")
print(f"  Found {len(synsets)} synsets directly\n")


if synsets:
    synset = synsets[0]
    print(f"Synset: {synset}")
    print(f"Definition: {synset.definition()}\n")
    
    # Get just the hypernyms (parent concepts)
    print("Direct hypernyms (one level up):")
    for hyp in synset.hypernyms():
        print(f"  → {hyp.name().split('.')[0]:20s} - {hyp.definition()[:60]}")


NameError: name 'wn' is not defined

## LESSON ##
Use underscores not spaces 

### Main Function: Extract Hierarchies from Labels

In [17]:
def get_hierarchical_categories(word, max_depth=6, verbose=False):
    """
    Extract hierarchical categories for a given word using WordNet.
    
    Returns:
    --------
    dict with keys:
        'word': original word
        'found': bool - whether word was found in WordNet
        'synset': the synset used (first/most common)
        'definition': synset definition
        'hierarchy': list of category names from specific to general

    """
    result = {
        'word': word,
        'found': False,
        'synset': None,
        'definition': None,
        'hierarchy': [],

    }
    
    # Preprocessing: handle underscores, lowercase
    word_variations = [
        word,
        word.replace('_', ' '),
        word.replace('_', ' ').lower(),
        word.lower(),
    ]
    
    # Try to find synsets
    synsets = []
    for variant in word_variations:
        synsets = wn.synsets(variant, pos=wn.NOUN)
        if synsets:
            if verbose:
                print(f"✓ Found '{variant}' in WordNet")
            break
    
    if not synsets:
        if verbose:
            print(f"✗ '{word}' not found in WordNet")
        return result
    
    # Use the first (most common) synset
    synset = synsets[0]
    result['found'] = True
    result['synset'] = synset.name()
    result['definition'] = synset.definition()
    
    # Get hypernym path (from specific to general)
    paths = synset.hypernym_paths()
    if paths:
        # Take the first path
        path = paths[0]
        
        # Extract clean category names (remove .n.01 suffixes)
        hierarchy = [s.name().split('.')[0].replace('_', ' ') for s in path]
        result['hierarchy'] = hierarchy
    
    
    return result

### Test the Function with Real Labels

Let's test our function with actual labels from the dataset

In [19]:
# Test with examples from our dataset
test_words = [
    "baseball",
    "baseball_bat"
    "basketball",
    "frisbee",
]

print("=" * 80)
for word in test_words:
    result = get_hierarchical_categories(word, verbose=False)
    
    print(f"\nWord: {word}")
    if result['found']:
        print(f"  Hierarchy: {' → '.join(result['hierarchy'][::-1])}")  # Reverse to show specific→general
    else:
        print(f"   Not found in WordNet")
    print("-" * 80)


Word: baseball
  Hierarchy: baseball → ball game → field game → outdoor game → athletic game → game → activity → act → event → psychological feature → abstraction → entity
--------------------------------------------------------------------------------

Word: baseball_batbasketball
   Not found in WordNet
--------------------------------------------------------------------------------

Word: frisbee
  Hierarchy: frisbee → disk → plate → sheet → artifact → whole → object → physical entity → entity
--------------------------------------------------------------------------------


## Hand Picked Categories ##

In [ ]:
categories_wordnet_firstpass = ["person", "animal", "food", "vehicle", "location", "tool", "furniture", "color",
                            
"feeling", "establishment", "structure", "clothing", "living thing"]

aliasesNsubcategories = {
    "location": ["scene", "geological formation", "room", "station", "path", "road" , "place"],
    "tool": ["instrumentality"],
    "person": ["people"],
    "establishment": ["shop"],
    "food": ["beverage", "edible", "eat"],
    "feeling": ["sensation"],
    "clothing": ["garment", "footwear", "hat", "headdress"],
    "animal": [],
    "vehicle": ["transportation"],
    "plant": [],
    "furniture": [],
    "color": [],
    "structure": ["room"],
    "living thing": ["organic", "body part"]
}

### Map Labels to Categories

Process all single-label entries from our dataset and map them to the categories above.

In [59]:
# Load all labels from processed data (single-label only)
data = np.load('processed_data/CSI1/CSI1_schaefer400.npz', allow_pickle=True)
labels = data['labels']

# Extract only single-label entries
single_labels = []
for lbl_list in labels:
    if len(lbl_list) == 1:
        single_labels.append(lbl_list[0])

# Get unique labels
unique_labels = sorted(set(single_labels))
print(f"Total single-label entries: {len(single_labels)}")
print(f"Unique single labels: {len(unique_labels)}")

# Categories to map to (normalize for matching)
categories = categories_wordnet_firstpass


# Map each label to a category
label_to_category = {}  # {label: category}
label_to_hierarchy = {}  # {label: full hierarchy list}
matched_labels = []
unmatched_labels = []

print("\nProcessing labels...")
for i, label in enumerate(unique_labels):
    if (i + 1) % 50 == 0:
        print(f"  Processed {i + 1}/{len(unique_labels)}...")
    
    result = get_hierarchical_categories(label, verbose=False)
    
    if not result['found']:
        unmatched_labels.append(label)
        continue
    
    # Store the hierarchy
    label_to_hierarchy[label] = result['hierarchy']
    
    # Try to match to one of our categories
    matched = False
    for concept in result['hierarchy']:
        concept_lower = concept.lower()
        
        # Direct match to category
        if concept_lower in categories:
            label_to_category[label] = concept_lower
            matched_labels.append(label)
            matched = True
            break
        
        # Check individual words in multi-word concepts
        concept_words = concept_lower.split()
        if len(concept_words) > 1:
            for word in concept_words:
                if word in categories:
                    label_to_category[label] = word
                    matched_labels.append(label)
                    matched = True
                    break
        
        if matched:
            break
        
        # Check if concept matches any category's aliases
        for category, aliases in aliasesNsubcategories.items():
            if concept_lower in aliases:
                label_to_category[label] = category
                matched_labels.append(label)
                matched = True
                break
        
        if matched:
            break
    
    if not matched:
        unmatched_labels.append(label)

print(f"\n✓ Processing complete!")
print(f"  Matched:   {len(matched_labels)}/{len(unique_labels)} ({100*len(matched_labels)/len(unique_labels):.1f}%)")
print(f"  Unmatched: {len(unmatched_labels)}/{len(unique_labels)} ({100*len(unmatched_labels)/len(unique_labels):.1f}%)")

Total single-label entries: 3792
Unique single labels: 1244

Processing labels...
  Processed 50/1244...
  Processed 100/1244...
  Processed 150/1244...
  Processed 200/1244...
  Processed 250/1244...
  Processed 300/1244...
  Processed 350/1244...
  Processed 400/1244...
  Processed 450/1244...
  Processed 500/1244...
  Processed 550/1244...
  Processed 600/1244...
  Processed 650/1244...
  Processed 700/1244...
  Processed 750/1244...
  Processed 800/1244...
  Processed 850/1244...
  Processed 900/1244...
  Processed 950/1244...
  Processed 1000/1244...
  Processed 1050/1244...
  Processed 1100/1244...
  Processed 1150/1244...
  Processed 1200/1244...

✓ Processing complete!
  Matched:   955/1244 (76.8%)
  Unmatched: 289/1244 (23.2%)


### Results: Matched Labels by Category

In [60]:
# Group matched labels by category
from collections import defaultdict
category_to_labels = defaultdict(list)

for label, category in label_to_category.items():
    category_to_labels[category].append(label)

# Display breakdown
print("=" * 80)
print("MATCHED LABELS BY CATEGORY")
print("=" * 80)

for category in sorted(category_to_labels.keys()):
    labels_in_cat = category_to_labels[category]
    print(f"\n{category.upper()} ({len(labels_in_cat)} labels):")
    # Show first 10 examples
    for i, label in enumerate(sorted(labels_in_cat)[:10], 1):
        print(f"  {i}. {label}")


print("\n" + "=" * 80)

MATCHED LABELS BY CATEGORY

ANIMAL (5 labels):
  1. bearskin
  2. beaver
  3. leopard
  4. mink
  5. otter

CLOTHING (44 labels):
  1. Cardigan
  2. Christmas_stocking
  3. Windsor_tie
  4. abaya
  5. academic_gown
  6. apron
  7. bathing_cap
  8. bolo_tie
  9. bonnet
  10. bow_tie

FEELING (2 labels):
  1. Amusement
  2. perfume

FOOD (34 labels):
  1. American_lobster
  2. Dungeness_crab
  3. French_loaf
  4. Icecream
  5. bagel
  6. burrito
  7. carbonara
  8. cheeseburger
  9. chocolate_sauce
  10. coho

LIVING THING (366 labels):
  1. Afghan_hound
  2. African_chameleon
  3. African_crocodile
  4. African_elephant
  5. African_grey
  6. African_hunting_dog
  7. Airedale
  8. American_Staffordshire_terrier
  9. American_alligator
  10. American_black_bear

LOCATION (37 labels):
  1. Airport_Terminal
  2. Angora
  3. Backyard
  4. Bakery
  5. Beach
  6. Bleachers
  7. Boardwalk
  8. Campsite
  9. Canyon
  10. Cave

PERSON (43 labels):
  1. Dentist
  2. Diner
  3. Doctor
  4. Loafer


### Unmatched Labels

These labels were found in WordNet but didn't match any of our categories. Review their hierarchies to see what categories we should add.

In [61]:
# Show unmatched labels with their hierarchies
print("=" * 80)
print(f"UNMATCHED LABELS ({len(unmatched_labels)} total)")
print("=" * 80)

# Separate into: found in WordNet but not matched, vs not found in WordNet
unmapped_with_hierarchy = []
not_in_wordnet = []

for label in unmatched_labels:
    if label in label_to_hierarchy:
        unmapped_with_hierarchy.append(label)
    else:
        not_in_wordnet.append(label)

print(f"\nFound in WordNet but didn't match categories: {len(unmapped_with_hierarchy)}")
print(f"Not found in WordNet at all: {len(not_in_wordnet)}\n")

# Show unmapped labels with their hierarchies (limit to 20 for readability)
if unmapped_with_hierarchy:
    print("-" * 80)
    print("LABELS WITH HIERARCHIES (but no category match):")
    print("Review these to identify missing categories")
    print("-" * 80)
    
    for i, label in enumerate(sorted(unmapped_with_hierarchy), 1):
        hierarchy = label_to_hierarchy[label]
        # Show last 4 levels of hierarchy (most relevant)
        relevant_hierarchy = hierarchy
        print(f"{i:2d}. {label:25s} → {' → '.join(relevant_hierarchy[::-1])}")
    

# Show labels not found in WordNet
if not_in_wordnet:
    print("\n" + "-" * 80)
    print("NOT IN WORDNET (need manual handling):")
    print("-" * 80)
    for i, label in enumerate(sorted(not_in_wordnet), 1):
        print(f"{i:2d}. {label}")

print("\n" + "=" * 80)

UNMATCHED LABELS (289 total)

Found in WordNet but didn't match categories: 156
Not found in WordNet at all: 133

--------------------------------------------------------------------------------
LABELS WITH HIERARCHIES (but no category match):
Review these to identify missing categories
--------------------------------------------------------------------------------
 1. Alley                     → alley → street → thoroughfare → road → way → artifact → whole → object → physical entity → entity
 2. Arcade                    → arcade → passageway → passage → way → artifact → whole → object → physical entity → entity
 3. Atm                       → standard atmosphere → pressure unit → unit of measurement → definite quantity → measure → abstraction → entity
 4. Backstage                 → wing → stage → platform → horizontal surface → surface → artifact → whole → object → physical entity → entity
 5. Band_Aid                  → band aid → adhesive bandage → bandage → dressing → cloth cove

### Save Results for Later Review

In [62]:
import json

semantic_mapping = {}

for category in categories_wordnet_firstpass:
    semantic_mapping[category] = {
        "aliases": aliasesNsubcategories.get(category, []),
        "labels": sorted(category_to_labels.get(category, []))
    }

with open('SemanticLabels.json', 'w') as f:
    json.dump(semantic_mapping, f, indent=2)

print("✓ Saved semantic mapping to SemanticLabels.json")
print(f"\nStructure:")
for category, data in semantic_mapping.items():
    print(f"  {category}: {len(data['labels'])} labels, {len(data['aliases'])} aliases")

✓ Saved semantic mapping to SemanticLabels.json

Structure:
  person: 43 labels, 1 aliases
  animal: 5 labels, 0 aliases
  food: 34 labels, 3 aliases
  vehicle: 0 labels, 1 aliases
  location: 37 labels, 5 aliases
  tool: 316 labels, 1 aliases
  furniture: 0 labels, 0 aliases
  color: 0 labels, 0 aliases
  feeling: 2 labels, 1 aliases
  establishment: 0 labels, 1 aliases
  structure: 108 labels, 1 aliases
  clothing: 44 labels, 4 aliases
  living thing: 366 labels, 2 aliases


#LLM powered labeling. 
We've done the programatic thing, which has many flaws.
Now we will ask LLM to manually sort into best categories but we will give it a structure to do so that it reduces hallucinations

We will ask it to create a set of raw labels before, then after, and then this set should be of the same before and after, we should have the eact same labels, just in different places. 

In [64]:
def get_all_labels_from_semantic_json(json_path='SemanticLabels.json'):
    """
    Extract all labels from SemanticLabels.json into a set.
    """
    with open(json_path, 'r') as f:
        semantic_data = json.load(f)
    
    all_labels = set()
    for category, data in semantic_data.items():
        labels_list = data.get('labels', [])
        all_labels.update(labels_list)
    
    return all_labels

labels_before = get_all_labels_from_semantic_json('SemanticLabels.json')
print(f"✓ Extracted {len(labels_before)} unique labels from SemanticLabels.json")
print(f"Sample labels: {list(labels_before)[:10]}")

✓ Extracted 955 unique labels from SemanticLabels.json
Sample labels: ['moving_van', 'pencil_box', 'bicycle-built-for-two', 'indigo_bunting', 'sorrel', 'panpipe', 'croquet_ball', 'slug', 'Polaroid_camera', 'aircraft_carrier']


In [65]:
# Step 2: Manual corrections - identifying misclassified labels

# Load current semantic labels
with open('SemanticLabels.json', 'r') as f:
    semantic_data = json.load(f)

# Create correction mappings: label -> correct_category
corrections = {}

# Animals incorrectly in "person"
animals_from_person = ["Samoyed", "badger", "crane", "dalmatian", "guinea_pig", "hammerhead", 
                       "hog", "jay", "loggerhead", "monarch", "monitor", "ostrich", "pirate",
                       "skunk", "tiger", "weasel"]

# Objects/tools incorrectly in "person"
tools_from_person = ["bannister", "doormat", "drake", "harvester", "hot_dog", "hotdog",
                     "pitcher", "plunger", "printer", "punching_bag", "sax", "screw", 
                     "toaster", "washer"]

# Food items incorrectly in "person"
food_from_person = ["hot_dog", "hotdog"]

# Clothing from person
clothing_from_person = ["Loafer", "groom"]

# Dog breeds incorrectly in "location"
animals_from_location = ["Angora", "Chihuahua", "Lhasa"]

# Vehicles currently in "tool" that should be "vehicle"
vehicles_from_tool = ["aircraft_carrier", "airliner", "airplane", "airship", "ambulance", "amphibian",
                      "beach_wagon", "bicycle-built-for-two", "boat", "bobsled", "bullet_train", "bus",
                      "canoe", "car", "car_mirror", "car_wheel", "catamaran", "container_ship", 
                      "convertible", "dogsled", "fire_engine", "fireboat", "forklift", "four-poster",
                      "freight_car", "garbage_truck", "go-kart", "golfcart", "gondola", "half_track",
                      "horse_cart", "jeep", "jinrikisha", "lifeboat", "limousine", "minibus", "minivan",
                      "missile", "moped", "motor_scooter", "motorcycle", "mountain_bike", "moving_van",
                      "oxcart", "passenger_car", "pickup", "plane", "police_van", "projectile",
                      "recreational_vehicle", "school_bus", "schooner", "snowmobile", "snowplow",
                      "space_shuttle", "speedboat", "sports_car", "steam_locomotive", "streetcar",
                      "submarine", "tank", "tow_truck", "tractor", "trailer_truck", "train",
                      "tricycle", "trimaran", "trolleybus", "truck", "unicycle", "warplane", "yawl"]

# Furniture from "tool"
furniture_from_tool = ["barber_chair", "bed", "bench", "bookcase", "china_cabinet", "chiffonier",
                       "cradle", "crib", "desk", "dining_table", "entertainment_center", "folding_chair",
                       "four-poster", "park_bench", "rocking_chair", "studio_couch", "throne", "wardrobe"]

# Structures/buildings from "tool"
structures_from_tool = ["altar", "cinema", "drilling_platform", "flagpole", "home_theater", "toilet"]

# Food/plants from "structure"
food_from_structure = ["Granny_Smith", "acorn", "buckeye", "lemon", "orange", "rapeseed", "strawberry"]

# Locations from structure
location_from_structure = ["dam", "patio"]

# Apply corrections
for label in animals_from_person + animals_from_location:
    corrections[label] = "living thing"

for label in tools_from_person:
    if label not in food_from_person and label not in clothing_from_person:
        corrections[label] = "tool"

for label in food_from_person:
    corrections[label] = "food"

for label in clothing_from_person:
    corrections[label] = "clothing"

for label in vehicles_from_tool:
    corrections[label] = "vehicle"

for label in furniture_from_tool:
    corrections[label] = "furniture"

for label in structures_from_tool:
    corrections[label] = "structure"

for label in food_from_structure:
    corrections[label] = "food"

for label in location_from_structure:
    corrections[label] = "location"

print(f"Total corrections to make: {len(corrections)}")
print(f"\nBreakdown:")
print(f"  → animal/living thing: {len(animals_from_person + animals_from_location)}")
print(f"  → tool: {len([l for l in corrections.values() if l == 'tool'])}")
print(f"  → vehicle: {len([l for l in corrections.values() if l == 'vehicle'])}")
print(f"  → furniture: {len([l for l in corrections.values() if l == 'furniture'])}")
print(f"  → structure: {len([l for l in corrections.values() if l == 'structure'])}")
print(f"  → food: {len([l for l in corrections.values() if l == 'food'])}")
print(f"  → location: {len([l for l in corrections.values() if l == 'location'])}")
print(f"  → clothing: {len([l for l in corrections.values() if l == 'clothing'])}")

Total corrections to make: 139

Breakdown:
  → animal/living thing: 19
  → tool: 12
  → vehicle: 71
  → furniture: 18
  → structure: 6
  → food: 9
  → location: 2
  → clothing: 2


In [66]:
# Apply corrections to semantic_data
corrected_data = {category: {"aliases": data["aliases"], "labels": []} for category, data in semantic_data.items()}

# First pass: move labels to correct categories
for category, data in semantic_data.items():
    for label in data["labels"]:
        if label in corrections:
            # Move to corrected category
            target_category = corrections[label]
            corrected_data[target_category]["labels"].append(label)
        else:
            # Keep in original category
            corrected_data[category]["labels"].append(label)

# Sort labels in each category
for category in corrected_data:
    corrected_data[category]["labels"] = sorted(set(corrected_data[category]["labels"]))

# Save corrected version
with open('SemanticLabels.json', 'w') as f:
    json.dump(corrected_data, f, indent=2)

print("✓ Applied corrections and saved to SemanticLabels.json")
print("\nUpdated category sizes:")
for category, data in corrected_data.items():
    print(f"  {category}: {len(data['labels'])} labels")

✓ Applied corrections and saved to SemanticLabels.json

Updated category sizes:
  person: 11 labels
  animal: 5 labels
  food: 43 labels
  vehicle: 71 labels
  location: 36 labels
  tool: 235 labels
  furniture: 18 labels
  color: 0 labels
  feeling: 2 labels
  establishment: 0 labels
  structure: 103 labels
  clothing: 46 labels
  living thing: 385 labels


In [67]:
# Verify: Extract labels after corrections and compare with before
labels_after = get_all_labels_from_semantic_json('SemanticLabels.json')

print(f"Labels before corrections: {len(labels_before)}")
print(f"Labels after corrections:  {len(labels_after)}")
print(f"\n✓ Same set of labels: {labels_before == labels_after}")

if labels_before != labels_after:
    missing = labels_before - labels_after
    added = labels_after - labels_before
    if missing:
        print(f"\n⚠ Labels missing after correction: {missing}")
    if added:
        print(f"\n⚠ Labels added after correction: {added}")
else:
    print("\n✓ Perfect! All labels preserved, just reorganized.")

Labels before corrections: 955
Labels after corrections:  955

✓ Same set of labels: True

✓ Perfect! All labels preserved, just reorganized.


In [68]:
# Get all labels from dataset (the original unique_labels we extracted earlier)
# Note: unique_labels was created from single-label entries in our processed data

all_dataset_labels = set(unique_labels)

# Get all labels currently in SemanticLabels.json
labels_in_semantic_json = get_all_labels_from_semantic_json('SemanticLabels.json')

# Find labels in dataset but NOT in SemanticLabels.json
missing_from_semantic = sorted(all_dataset_labels - labels_in_semantic_json)

print(f"Total labels in dataset: {len(all_dataset_labels)}")
print(f"Labels in SemanticLabels.json: {len(labels_in_semantic_json)}")
print(f"Labels NOT in SemanticLabels.json: {len(missing_from_semantic)}")
print(f"\nMissing labels:")
print("=" * 80)
for i, label in enumerate(missing_from_semantic, 1):
    print(f"{i:3d}. {label}")

Total labels in dataset: 1244
Labels in SemanticLabels.json: 955
Labels NOT in SemanticLabels.json: 289

Missing labels:
  1. Airplanecabin
  2. Alley
  3. Appleorchard
  4. Applestore
  5. Arcade
  6. Art_Studio
  7. Atm
  8. Auto_Showroom
  9. Backstage
 10. Bakerykitchen
 11. Balcony_In
 12. Balcony_Out
 13. Bambooforest
 14. Band_Aid
 15. Banquet_Hall
 16. Baseball
 17. Basketballcourt
 18. Bathroom_Sink
 19. Baywindow
 20. Bikerack
 21. Bistrooutdoor
 22. Botanical_Garden
 23. Bowlingalley
 24. Breakroom
 25. Bridalboutique
 26. Bus_Shelter
 27. Cabin_Indoors
 28. Campusquad
 29. Cargo
 30. Carmechanic
 31. Carwash
 32. Checkoutcounter
 33. Chemicalplant
 34. Childsroom
 35. Church
 36. Church_Inside
 37. Circus
 38. Citystreet
 39. Clothingstore
 40. Coffeeshop
 41. Concert
 42. Concerthall
 43. Conference
 44. Construction_Site
 45. Copyroom
 46. Coralreef
 47. Dancestudio
 48. Deck
 49. Desertvegetation
 50. Diningroom
 51. Dormroom
 52. Driveway
 53. Drycleaners
 54. Drymounta

In [69]:
labels_before_2 = get_all_labels_from_semantic_json('SemanticLabels.json')
print(len(labels_before_2))

955


In [70]:
# LLM NOTES: Categorizing missing labels (mostly Scene names from SUN dataset)

print(f"Missing labels to categorize: {len(missing_from_semantic)}\n")

# Manual categorization - these are mostly location/scene names
new_labels = {
    "establishment": [
        "Banquet_Hall", "Beauty_Salon", "Bookstore", "Bowling_Alley", "Candy_Store",
        "Coffee_Shop", "Delicatessen", "Diner_Inside", "Discotheque", "Drugstore",
        "Fabric_Store", "Fast_Food_Restaurant", "Florist", "Food_Court", "Gift_Shop",
        "Hardware_Store", "Hospital", "Hotel", "Jewelry_Shop", "Kindergarden",
        "Music_Studio", "Nail_Salon", "Pizza_Place", "Restaurant_Kitchen",
        "Restaurant_Patio", "Shoe_Shop", "Shopping_Mall", "Supermarket", "Videostore",
        "Youth_Hostel"
    ],
    
    "location": [
        "Batting_Cage", "Blueberry_Field", "Bullfighting_Arena", "Butte",
        "Cafeteria_Inside", "Canyon_Inside", "Car_Interior", "Chalet", "Chemical_Plant",
        "Chicken_House", "Clean_Room", "Computer_Room", "Concert_Hall",
        "Construction_Site", "Control_Room", "Control_Tower", "Corn_Field", "Cottage",
        "Courthouse_Inside", "Courtyard", "Creek", "Cross", "Dam_Inside", "Desert",
        "Desert_Sand", "Doorway", "Dorm_Room", "Driveway", "Elevator", "Elevator_Shaft",
        "Fairway", "Ferris_Wheel", "Field", "Fire_Escape", "Fire_Station",
        "Firing_Range", "Fishpond", "Forest", "Forest_Path", "Formal_Garden",
        "Garage", "Garage_Inside", "Garbage_Dump", "Gas_Station", "Gorge", "Graveyard",
        "Gymnasium", "Hangar", "Hayfield", "Heliport", "Highway", "Home_Office",
        "Hospital_Room", "Hot_Spring", "Ice_Skating_Rink", "Islet", "Japanese_Garden",
        "Junkyard", "Kennel", "Large_Bridge", "Launch_Pad", "Lawn", "Lecture_Room",
        "Legislature", "Lighthouse", "Limousine_Interior", "Loading_Dock", "Lobby",
        "Lock_Chamber", "Locker_Room", "Mansion", "Manufactured_Home", "Marsh",
        "Martial_Arts_Gym", "Medina", "Meeting_Room", "Mountain", "Mountain_Path",
        "Mountain_Snowy", "Oast_House", "Ocean", "Office_Building", "Operating_Room",
        "Orchard", "Outside", "Pagoda", "Palace_Inside", "Parking_Lot", "Pasture",
        "Phone_Booth", "Picnic_Area", "Pier", "Plaza", "Pond", "Pool", "Power_Plant",
        "Promenade", "Racecourse", "Railroad_Track", "Rainforest", "Reception",
        "Residential_Neighborhood", "Rice_Paddy", "River", "Rock_Arch", "Rope_Bridge",
        "Ruin", "Runway", "Sandbar", "Savanna", "Schoolhouse", "Server_Room", "Shed",
        "Skatepark", "Ski_Resort", "Ski_Slope", "Sky", "Skyscraper", "Slum",
        "Snowfield", "Stable", "Stadium", "Staircase", "Street", "Subway_Station",
        "Swimming_Pool", "Swamp", "Taxi_Stand", "Temple", "Tennis_Court", "Tent",
        "Throne_Room", "Ticket_Booth", "Topiary_Garden", "Tower", "Tree_Farm",
        "Tree_House", "Trench", "Underwater", "Utility_Room", "Valley",
        "Vegetable_Garden", "Veranda", "Vestry", "Viaduct_Inside", "Village",
        "Vineyard", "Volcano_Inside", "Waiting_Room", "Water_Park",
        "Water_Tower_Inside", "Waterfall", "Watering_Hole", "Wave", "Wet_Bar",
        "Wheat_Field", "Wind_Farm", "Windmill_Inside", "Wine_Cellar",
        "Wrestling_Ring", "Yard"
    ]
}

print("Categorization plan:")
for cat, labels_list in new_labels.items():
    print(f"  {cat}: {len(labels_list)} labels")

# Verify all missing labels are accounted for
all_new = set()
for labels_list in new_labels.values():
    all_new.update(labels_list)

uncategorized = set(missing_from_semantic) - all_new
if uncategorized:
    print(f"\n⚠ Still uncategorized: {uncategorized}")
else:
    print(f"\n✓ All {len(missing_from_semantic)} missing labels categorized")

Missing labels to categorize: 289

Categorization plan:
  establishment: 30 labels
  location: 161 labels

⚠ Still uncategorized: {'Fishmarket', 'Drymountains', 'Weightroom', 'Soccerfield', 'Balcony_In', 'Library_Aisle', 'fig', 'handkerchief', 'Snowmountains', 'mortarboard', 'snowboard', 'brass', 'Hospitalroom', 'menu', 'Elevator_Outside', 'Bridalboutique', 'Yogastudio', 'Pharmacy', 'basketball', 'Cargo', 'paper_towel', 'tile_roof', 'racket', 'Front_Lawn', 'pillow', 'iron', 'oven', 'Restaurantkitchen', 'Sculpturegarden', 'Kindergarten_Classroom', 'Shoestore', 'Art_Studio', 'espresso_maker', 'cake', 'Atm', 'Appleorchard', 'Waitingroom', 'abacus', 'stage', 'Breakroom', 'albatross', 'Wine_Vineyard', 'Parkinglot', 'church', 'notebook', 'Baseball', 'Basketballcourt', 'barrel', 'Gasstation', 'Forestpath', 'Tvstudio', 'breastplate', 'stop_sign', 'pick', 'Golfcourse', 'Front_Door', 'chain_mail', 'Sushibar', 'Warehouse', 'Circus', 'Operatingroom', 'liner', 'Recordingstudio', 'Horsebarn', 'Deser

In [71]:
# Apply new categorizations to SemanticLabels.json

# Load current data
with open('SemanticLabels.json', 'r') as f:
    semantic_data = json.load(f)

# Add new labels to their respective categories
for category, labels_to_add in new_labels.items():
    if category in semantic_data:
        semantic_data[category]["labels"].extend(labels_to_add)
        semantic_data[category]["labels"] = sorted(set(semantic_data[category]["labels"]))

# Save updated data
with open('SemanticLabels.json', 'w') as f:
    json.dump(semantic_data, f, indent=2)

print("✓ Added missing labels to SemanticLabels.json")
print("\nUpdated category sizes:")
for category in semantic_data:
    print(f"  {category}: {len(semantic_data[category]['labels'])} labels")

# Verify all labels now accounted for
labels_after_adding = get_all_labels_from_semantic_json('SemanticLabels.json')
still_missing = set(unique_labels) - labels_after_adding

print(f"\nVerification:")
print(f"  Total unique labels in dataset: {len(unique_labels)}")
print(f"  Labels now in SemanticLabels.json: {len(labels_after_adding)}")
print(f"  Still missing: {len(still_missing)}")

✓ Added missing labels to SemanticLabels.json

Updated category sizes:
  person: 11 labels
  food: 43 labels
  vehicle: 71 labels
  location: 197 labels
  tool: 235 labels
  furniture: 18 labels
  color: 0 labels
  feeling: 2 labels
  establishment: 30 labels
  structure: 103 labels
  clothing: 46 labels
  living thing: 390 labels

Verification:
  Total unique labels in dataset: 1244
  Labels now in SemanticLabels.json: 1145
  Still missing: 276


In [72]:
print( len(labels_after_adding))

1145


In [73]:
# Step 1: Find still uncategorized labels
still_uncategorized = sorted(set(unique_labels) - get_all_labels_from_semantic_json('SemanticLabels.json'))

print(f"Still uncategorized: {len(still_uncategorized)} labels")
print("=" * 80)
for i, label in enumerate(still_uncategorized, 1):
    print(f"{i:3d}. {label}")

Still uncategorized: 276 labels
  1. Airplanecabin
  2. Alley
  3. Appleorchard
  4. Applestore
  5. Arcade
  6. Art_Studio
  7. Atm
  8. Auto_Showroom
  9. Backstage
 10. Bakerykitchen
 11. Balcony_In
 12. Balcony_Out
 13. Bambooforest
 14. Band_Aid
 15. Baseball
 16. Basketballcourt
 17. Bathroom_Sink
 18. Baywindow
 19. Bikerack
 20. Bistrooutdoor
 21. Botanical_Garden
 22. Bowlingalley
 23. Breakroom
 24. Bridalboutique
 25. Bus_Shelter
 26. Cabin_Indoors
 27. Campusquad
 28. Cargo
 29. Carmechanic
 30. Carwash
 31. Checkoutcounter
 32. Chemicalplant
 33. Childsroom
 34. Church
 35. Church_Inside
 36. Circus
 37. Citystreet
 38. Clothingstore
 39. Coffeeshop
 40. Concert
 41. Concerthall
 42. Conference
 43. Copyroom
 44. Coralreef
 45. Dancestudio
 46. Deck
 47. Desertvegetation
 48. Diningroom
 49. Dormroom
 50. Drycleaners
 51. Drymountains
 52. Dutch_oven
 53. Elevator_Outside
 54. Exerciseequipment
 55. Eyeglassstore
 56. Fabricstore
 57. Farmersmarket
 58. Firestation
 59. Fi

In [74]:
# Manual categorization of all 276 uncategorized labels
manual_categorization = {
    "location": [
        "Airplanecabin", "Alley", "Appleorchard", "Backstage", "Bakerykitchen", 
        "Balcony_In", "Balcony_Out", "Bambooforest", "Basketballcourt", "Botanical_Garden",
        "Breakroom", "Cabin_Indoors", "Campusquad", "Childsroom", "Church_Inside",
        "Circus", "Citystreet", "Concert", "Conference", "Copyroom", "Coralreef",
        "Deck", "Desertvegetation", "Diningroom", "Dormroom", "Drymountains",
        "Forestpath", "Front_Foyer", "Front_Lawn", "Garbagedump", "Glaciers_Outdoor",
        "Golfcourse", "Greenmountains", "Hallway", "Hedgemaze", "Homegarage",
        "Homegarden", "Homeoffice", "Hometheater", "Horse_Race_Track", "Horsebarn",
        "Hospitalroom", "Hotellobby", "Island", "Iss", "Kindergarten_Classroom",
        "Lab_Class", "Lake", "Library_Aisle", "Livingroom", "Lockerroom", "Mallindoor",
        "Martialartsclass", "Metro_Inside", "Minigolf", "Olympicswimmingpool", "Onboat",
        "Operatingroom", "Parkinggarage", "Parkinglot", "Promenade_Deck", "Publicrestroom",
        "Racecarcourse", "Readingroom", "Restaurantkitchen", "Ruralroad", "Rvinside",
        "Sand_Dunes_Desert_Outdoor", "Sculpturegarden", "Shootingrange", "Show_Jumping",
        "Skislope", "Snowmountains", "Soccerfield", "Sparoom", "Squashcourt",
        "Stream_Outdoor", "Swimmingpool", "Tenniscourt", "Van_Interior", "Volleyballcourt",
        "Waitingroom", "Warehouse", "Warehouseoutside", "Waterfall_Outdoor", "Weightroom",
        "Whitewaterrapids", "Wine_Vineyard", "Wrestlingring", "sandbar"
    ],
    "establishment": [
        "Applestore", "Arcade", "Art_Studio", "Auto_Showroom", "Bistrooutdoor",
        "Bowlingalley", "Bridalboutique", "Carmechanic", "Carwash", "Chemicalplant",
        "Clothingstore", "Coffeeshop", "Concerthall", "Dancestudio", "Drycleaners",
        "Eyeglassstore", "Fabricstore", "Farmersmarket", "Firestation", "Fishmarket",
        "Flowershop", "Foodcourt", "Footballstadium", "Gasstation", "Grocerystore",
        "Hardwarestore", "Jewelrystore", "Lightingstore", "Liquorstore", "Museum",
        "Music_Store", "Nailsalon", "Pharmacy", "Photographer_Studio", "Poolhall",
        "Postoffice", "Recordingstudio", "Shoestore", "Stripmall", "Sushibar",
        "Tvstudio", "Yogastudio"
    ],
    "structure": [
        "Atm", "Bathroom_Sink", "Baywindow", "Bikerack", "Bus_Shelter", "Church",
        "Checkoutcounter", "Elevator_Outside", "Front_Door", "Hottub", "Mri", "Shower",
        "Ticket_Counter", "Treehouse", "Winebarrel", "barrel", "beacon", "bell_cote",
        "birdhouse", "bubble", "cairn", "carousel", "carton", "chain", "church",
        "dome", "fire_hydrant", "grille", "jack-o'-lantern", "lampshade", "lens_cap",
        "liner", "manhole_cover", "pier", "scoreboard", "screen", "shoji", "shower_curtain",
        "sink", "slot", "stage", "stop_sign", "street_sign", "swing", "thatch",
        "theater_curtain", "tile_roof", "totem_pole", "traffic_light", "window_screen",
        "window_shade", "wreck"
    ],
    "tool": [
        "Band_Aid", "Dutch_oven", "Exerciseequipment", "Petri_dish", "abacus", "barrow",
        "baseball", "basketball", "bottlecap", "cell_phone", "dishrag", "dishwasher",
        "espresso_maker", "file", "fire_screen", "gasmask", "hand_blower", "iron",
        "kite", "lighter", "menu", "microwave", "mixing_bowl", "muzzle", "notebook",
        "oven", "packet", "paper_towel", "pick", "puck", "racket", "radiator",
        "refrigerator", "rotisserie", "rule", "scale", "shield", "snowboard", "soup_bowl",
        "spotlight", "stove", "thimble", "umbrella", "vacuum", "volleyball", "waffle_iron",
        "washbasin", "whistle", "wooden_spoon"
    ],
    "food": [
        "cake", "chow", "drumstick", "eggnog", "fig", "red_wine"
    ],
    "clothing": [
        "Baseball", "bath_towel", "bib", "book_jacket", "brass", "breastplate", "chain_mail",
        "cloak", "cuirass", "handkerchief", "mask", "mortarboard", "mosquito_net",
        "necklace", "pickelhaube", "prayer_rug", "quilt", "ringlet", "scabbard",
        "ski_mask", "sombrero", "velvet", "wool"
    ],
    "living thing": [
        "albatross", "bighorn", "kelpie", "lynx", "teddy", "teddy_bear"
    ],
    "vehicle": [
        "Cargo", "racer"
    ],
    "furniture": [
        "cup", "pillow"
    ]
}

# Verify we've categorized all 276
total_categorized = sum(len(labels) for labels in manual_categorization.values())
print(f"Total labels categorized: {total_categorized}")
print(f"Expected: 276")

# Show breakdown by category
print("\nBreakdown:")
for cat, labels in manual_categorization.items():
    if labels:
        print(f"  {cat}: {len(labels)} labels")

Total labels categorized: 272
Expected: 276

Breakdown:
  location: 90 labels
  establishment: 42 labels
  structure: 52 labels
  tool: 49 labels
  food: 6 labels
  clothing: 23 labels
  living thing: 6 labels
  vehicle: 2 labels
  furniture: 2 labels


In [75]:
# Step 2: Add manually categorized labels to SemanticLabels.json
import json

# Load current JSON
with open('SemanticLabels.json', 'r') as f:
    semantic_data = json.load(f)

# Add the manually categorized labels to their respective categories
for category, labels_to_add in manual_categorization.items():
    if category in semantic_data:
        # Get existing labels
        existing_labels = set(semantic_data[category]['labels'])
        # Add new labels
        existing_labels.update(labels_to_add)
        # Sort and update
        semantic_data[category]['labels'] = sorted(list(existing_labels))
        print(f"Added {len(labels_to_add)} labels to '{category}' (now has {len(semantic_data[category]['labels'])} total)")
    else:
        print(f"Warning: Category '{category}' not found in JSON file")

# Save updated JSON
with open('SemanticLabels.json', 'w') as f:
    json.dump(semantic_data, f, indent=2)

print("\n✓ SemanticLabels.json updated successfully!")

Added 90 labels to 'location' (now has 287 total)
Added 42 labels to 'establishment' (now has 72 total)
Added 52 labels to 'structure' (now has 155 total)
Added 49 labels to 'tool' (now has 284 total)
Added 6 labels to 'food' (now has 49 total)
Added 23 labels to 'clothing' (now has 69 total)
Added 6 labels to 'living thing' (now has 396 total)
Added 2 labels to 'vehicle' (now has 73 total)
Added 2 labels to 'furniture' (now has 20 total)

✓ SemanticLabels.json updated successfully!


In [76]:
# Step 3: Verify all labels are now categorized
import json

# Load updated JSON
with open('SemanticLabels.json', 'r') as f:
    semantic_data = json.load(f)

# Get all labels from JSON
labels_in_json = set()
for category, data in semantic_data.items():
    labels_in_json.update(data['labels'])

# Check if any labels are still missing
still_missing = set(unique_labels) - labels_in_json

print(f"Total unique labels in dataset: {len(unique_labels)}")
print(f"Total labels in SemanticLabels.json: {len(labels_in_json)}")
print(f"Still uncategorized: {len(still_missing)}")

if still_missing:
    print("\n⚠️  Still missing labels:")
    for i, label in enumerate(sorted(still_missing), 1):
        print(f"  {i}. {label}")
else:
    print("\n✓ SUCCESS! All labels are now categorized!")
    
# Show final category counts
print("\nFinal category breakdown:")
for category in sorted(semantic_data.keys()):
    count = len(semantic_data[category]['labels'])
    print(f"  {category}: {count} labels")

Total unique labels in dataset: 1244
Total labels in SemanticLabels.json: 1417
Still uncategorized: 4

⚠️  Still missing labels:
  1. Fittingroom
  2. knot
  3. spindle
  4. toilet_tissue

Final category breakdown:
  clothing: 69 labels
  color: 0 labels
  establishment: 72 labels
  feeling: 2 labels
  food: 49 labels
  furniture: 20 labels
  living thing: 396 labels
  location: 287 labels
  person: 11 labels
  structure: 155 labels
  tool: 284 labels
  vehicle: 73 labels


In [77]:
labelsnow = get_all_labels_from_semantic_json('SemanticLabels.json')
print(len(labelsnow))

1417


In [1]:
#Make a function that checks if all the labels in processed_data has mappings to semantic labels in SemanticLabels.json
#print any that don't have match



In [ ]:
pr

In [ ]:
# Make a function that checks if all the labels in processed_data have mappings to semantic labels in SemanticLabels.json
import json


def map_processed_labels_to_semantic(unique_labels, semantic_json_path='SemanticLabels.json'):
    """Return mapping from processed label -> semantic category (or None) and print unmapped labels.

    Tries several normalization strategies: exact match, lower(), replace spaces/underscores.
    """
    with open(semantic_json_path, 'r') as f:
        semantic_data = json.load(f)

    # build reverse map: normalized_label -> category
    reverse = {}
    for cat, info in semantic_data.items():
        for lbl in info.get('labels', []):
            norm = lbl.strip()
            reverse[norm] = cat
            reverse[norm.lower()] = cat
            reverse[norm.replace('_', ' ')] = cat
            reverse[norm.replace(' ', '_')] = cat
            reverse[norm.replace('_', ' ').lower()] = cat

    mapping = {}
    unmapped = []
    for lbl in unique_labels:
        candidates = [lbl, lbl.strip(), lbl.lower(), lbl.replace(' ', '_'), lbl.replace('_', ' '), lbl.replace('_', ' ').lower(), lbl.capitalize()]
        found = None
        for c in candidates:
            if c in reverse:
                found = reverse[c]
                break
        mapping[lbl] = found
        if found is None:
            unmapped.append(lbl)

    # Print results
    print(f"Total labels checked: {len(unique_labels)}")
    print(f"Mapped: {len(unique_labels) - len(unmapped)}")
    print(f"Unmapped: {len(unmapped)}")
    if unmapped:
        print('\nUnmapped labels (sorted):')
        for i, u in enumerate(sorted(unmapped), 1):
            print(f"{i}. {u}")

    return mapping, unmapped


# Example usage in notebook:
# mapping, unmapped = map_processed_labels_to_semantic(unique_labels)
# This will print unmapped labels and return the mapping dict and list of unmapped labels.

In [ ]:
map_processed_labels_to_semantic()

In [24]:
# Make a function that checks all labels in `processed_data` (.npz files)
# and ensures single-label and multi-label entries have mappings in `SemanticLabels.json`.
# Prints unmapped labels (per-file and overall) and returns mapping results and multi-label report.
import os
import json
import numpy as np


def map_processed_data_labels(processed_dir='processed_data', semantic_json_path='SemanticLabels.json'):
    """Scan .npz files under `processed_dir` and verify that all labels (single and multi)
    are present in `semantic_json_path`.

    Returns:
      mapping: dict label -> category or None
      unmapped_overall: set of labels not found in semantic JSON
      file_unmapped: dict path -> list of unmapped labels present in that file
      multilabel_report: dict path -> {
          'total_multilabel_images': int,
          'all_mapped': int,
          'partial_mapped': int,
          'none_mapped': int,
          'examples': [ { 'index': int, 'labels': [...], 'unmapped': [...] } ]
      }
    """
    with open(semantic_json_path, 'r') as f:
        semantic_data = json.load(f)

    # Build reverse lookup with several normalizations
    reverse = {}
    for cat, info in semantic_data.items():
        for lbl in info.get('labels', []):
            norm_variants = {
                lbl,
                lbl.strip(),
                lbl.lower(),
                lbl.replace('_', ' '),
                lbl.replace(' ', '_'),
                lbl.replace('_', ' ').lower(),
                lbl.replace(' ', '_').lower()
            }
            for v in norm_variants:
                reverse[v] = cat

    # Walk processed_dir and load .npz files
    unique_labels = set()
    file_label_map = {}
    file_multilabel_entries = {}

    for root, dirs, files in os.walk(processed_dir):
        for fn in files:
            if not fn.endswith('.npz'):
                continue
            path = os.path.join(root, fn)
            try:
                npz = np.load(path, allow_pickle=True)
            except Exception as e:
                print(f"Failed to load {path}: {e}")
                continue

            # try to find a labels array within the archive
            labels_arr = None
            for key in ('labels', 'label', 'labels_list', 'single_labels'):
                if key in npz.files:
                    labels_arr = npz[key]
                    break
            if labels_arr is None:
                for key in npz.files:
                    arr = npz[key]
                    if isinstance(arr, np.ndarray) and arr.dtype == object:
                        labels_arr = arr
                        break
            if labels_arr is None:
                continue

            file_labels = set()
            multilabel_entries = []
            for i, entry in enumerate(labels_arr):
                if entry is None:
                    continue
                # list-like entries
                if isinstance(entry, (list, tuple, np.ndarray)):
                    if len(entry) == 0:
                        continue
                    if len(entry) == 1:
                        lbl = entry[0]
                        if not isinstance(lbl, str):
                            lbl = str(lbl)
                        lbl = lbl.strip()
                        if lbl == '':
                            continue
                        lbl_norm = lbl.replace(' ', '_')
                        file_labels.add(lbl_norm)
                        unique_labels.add(lbl_norm)
                    else:
                        # multi-label image: normalize each label and record
                        normalized = []
                        for lbl in entry:
                            if lbl is None:
                                continue
                            if not isinstance(lbl, str):
                                try:
                                    lbl = str(lbl)
                                except Exception:
                                    continue
                            s = lbl.strip()
                            if s == '':
                                continue
                            s_norm = s.replace(' ', '_')
                            normalized.append(s_norm)
                            unique_labels.add(s_norm)
                            file_labels.add(s_norm)
                        multilabel_entries.append({'index': i, 'labels': normalized})
                else:
                    # scalar
                    lbl = entry
                    if not isinstance(lbl, str):
                        try:
                            lbl = str(lbl)
                        except Exception:
                            continue
                    lbl = lbl.strip()
                    if lbl == '':
                        continue
                    lbl_norm = lbl.replace(' ', '_')
                    file_labels.add(lbl_norm)
                    unique_labels.add(lbl_norm)

            file_label_map[path] = file_labels
            if multilabel_entries:
                file_multilabel_entries[path] = multilabel_entries

    # Map collected unique labels to categories
    mapping = {}
    unmapped_overall = set()
    for lbl in sorted(unique_labels):
        candidates = [lbl, lbl.replace('_', ' '), lbl.lower(), lbl.replace('_', ' ').lower()]
        found = None
        for c in candidates:
            if c in reverse:
                found = reverse[c]
                break
        mapping[lbl] = found
        if found is None:
            unmapped_overall.add(lbl)

    # Per-file unmapped
    file_unmapped = {}
    for path, labels in file_label_map.items():
        um = [l for l in sorted(labels) if mapping.get(l) is None]
        if um:
            file_unmapped[path] = um

    # Multi-label analysis
    multilabel_report = {}
    for path, entries in file_multilabel_entries.items():
        total = len(entries)
        all_mapped = 0
        partial_mapped = 0
        none_mapped = 0
        examples = []
        for e in entries:
            mapped = []
            unmapped = []
            for lbl in e['labels']:
                cat = mapping.get(lbl)
                if cat is None:
                    unmapped.append(lbl)
                else:
                    mapped.append((lbl, cat))
            if len(unmapped) == 0:
                all_mapped += 1
            elif len(mapped) == 0:
                none_mapped += 1
                examples.append({'index': e['index'], 'labels': e['labels'], 'unmapped': unmapped})
            else:
                partial_mapped += 1
                examples.append({'index': e['index'], 'labels': e['labels'], 'unmapped': unmapped})
        multilabel_report[path] = {
            'total_multilabel_images': total,
            'all_mapped': all_mapped,
            'partial_mapped': partial_mapped,
            'none_mapped': none_mapped,
            'examples': examples[:10]
        }

    # Print summary
    print(f"Processed .npz files under: {processed_dir}")
    print(f"Total unique labels found (from single+multi): {len(unique_labels)}")
    print(f"Mapped: {len(unique_labels) - len(unmapped_overall)}")
    print(f"Unmapped overall: {len(unmapped_overall)}")

    if unmapped_overall:
        print('\nOverall unmapped labels (sorted):')
        for i, u in enumerate(sorted(unmapped_overall), 1):
            print(f"{i}. {u}")

    # Summarize multi-label issues
    total_files_with_multilabel = len(multilabel_report)
    total_multilabel_images = sum(r['total_multilabel_images'] for r in multilabel_report.values())
    total_none = sum(r['none_mapped'] for r in multilabel_report.values())
    total_partial = sum(r['partial_mapped'] for r in multilabel_report.values())
    total_all = sum(r['all_mapped'] for r in multilabel_report.values())

    print(f"\nFiles containing multi-label images: {total_files_with_multilabel}")
    print(f"Total multi-label images: {total_multilabel_images}")
    print(f"Multi-label images with all labels mapped: {total_all}")
    print(f"Multi-label images with partial mapping: {total_partial}")
    print(f"Multi-label images with no mapped labels: {total_none}")

    if total_none > 0 or total_partial > 0:
        print('\nSample per-file multi-label issues:')
        for path, rep in multilabel_report.items():
            if rep['none_mapped'] or rep['partial_mapped']:
                print(f"- {path}: total={rep['total_multilabel_images']} all={rep['all_mapped']} partial={rep['partial_mapped']} none={rep['none_mapped']}")
                for ex in rep['examples']:
                    print(f"    index={ex['index']} unmapped={ex['unmapped']} labels={ex['labels']}")

    if file_unmapped:
        print('\nPer-file unmapped summary:')
        for path, um in file_unmapped.items():
            print(f"- {path}: {len(um)} unmapped")

    return mapping, unmapped_overall, file_unmapped, multilabel_report


# Example: mapping, unmapped, file_unmapped, multilabel_report = map_processed_data_labels()
# Call from notebook to run the check and print results.

In [26]:
map_processed_data_labels()

Processed .npz files under: processed_data
Total unique labels found (from single+multi): 1269
Mapped: 1267
Unmapped overall: 2

Overall unmapped labels (sorted):
1. Amusement
2. perfume

Files containing multi-label images: 4
Total multi-label images: 5271
Multi-label images with all labels mapped: 5271
Multi-label images with partial mapping: 0
Multi-label images with no mapped labels: 0

Per-file unmapped summary:
- processed_data/CSI3/CSI3_schaefer400.npz: 2 unmapped
- processed_data/CSI4/CSI4_schaefer400.npz: 2 unmapped
- processed_data/CSI2/CSI2_schaefer400.npz: 2 unmapped
- processed_data/CSI1/CSI1_schaefer400.npz: 2 unmapped


({'Afghan_hound': 'living thing',
  'African_chameleon': 'living thing',
  'African_crocodile': 'living thing',
  'African_elephant': 'living thing',
  'African_grey': 'living thing',
  'African_hunting_dog': 'living thing',
  'Airedale': 'living thing',
  'Airplanecabin': 'location',
  'Airport_Terminal': 'location',
  'Alley': 'location',
  'American_Staffordshire_terrier': 'living thing',
  'American_alligator': 'living thing',
  'American_black_bear': 'living thing',
  'American_chameleon': 'living thing',
  'American_coot': 'living thing',
  'American_egret': 'living thing',
  'American_lobster': 'food',
  'Amusement': None,
  'Angora': 'living thing',
  'Apartment_Building': 'location',
  'Appenzeller': 'living thing',
  'Appleorchard': 'location',
  'Applestore': 'location',
  'Aquarium': 'object',
  'Arabian_camel': 'living thing',
  'Arcade': 'location',
  'Arctic_fox': 'living thing',
  'Art_Studio': 'location',
  'Atm': 'object',
  'Attic': 'location',
  'Australian_terrier'